<a href="https://colab.research.google.com/github/Neloy-Barman/Fast-Efficient-LLM-Inference-with-vLLM/blob/main/LLM_Compression_From_Course.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Requirements Installation

### `uv` because its dependency conflict resolution is faster

In [1]:
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 88.2 MB/s  0:00:00


### Installing llm-compressor using uv

In [2]:
!uv pip install llmcompressor

Using Python 3.13.16 environment at: /usr
Resolved 87 packages in 1.68s
Prepared 6 packages in 1.09s
Uninstalled 2 packages in 912ms
Installed 6 packages in 68ms
 + auto-round==0.16.1
 + compressed-tensors==0.20.1
 + llmcompressor==0.15.0
 + loguru==0.8.0
 - pillow==11.3.0
 + pillow==13.0.0
 - transformers==5.22.1
 + transformers==5.21.0


## Mounting Google Drive

In [3]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Pre-create the target directory
DRIVE_BASE_DIR = "/content/drive/MyDrive/llm_compressor_practice"
os.makedirs(DRIVE_BASE_DIR, exist_ok=True)
print(f"Google Drive folder ready at: {DRIVE_BASE_DIR}")

Mounted at /content/drive
Google Drive folder ready at: /content/drive/MyDrive/llm_compressor_practice


## Checking GPU

In [4]:
!nvidia-smi

Fri Nov 20 11:50:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Base Model Dir

In [6]:
# MODEL_ID = "Qwen/Qwen3-30B-A3B"
MODEL_ID = "Qwen/Qwen3-0.6B"
BASE_MODEL_SAVE_DIR = os.path.join(DRIVE_BASE_DIR, f"{MODEL_ID}")
os.makedirs(BASE_MODEL_SAVE_DIR, exist_ok=True)

### Loading Base model from saved dir

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# # Load Model
# model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
# # Load Tokenizer
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# # Save model and tokenizer to the defined directory.
# model.save_pretrained(BASE_MODEL_SAVE_DIR)
# tokenizer.save_pretrained(BASE_MODEL_SAVE_DIR)

# Load model and tokenizer from saved dir
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_SAVE_DIR, torch_dtype="auto")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_SAVE_DIR)
print(f"Successfully loaded base model and tokenizer from: {BASE_MODEL_SAVE_DIR}")

Loading weights:   0%|          | 0/310 [00:02<?, ?it/s]

Successfully loaded base model and tokenizer from: /content/drive/MyDrive/llm_compressor_practice/Qwen/Qwen3-0.6B


## Compression Scheme and Quantization
- W4A16 - Targeting `Weights` to 4-bits
- `GPTQ`  

In [37]:
from llmcompressor.modifiers.quantization import GPTQModifier

recipe = GPTQModifier(
    scheme="W4A16",
    targets="Linear",
    ignore=["lm_head"],
)

print(f"Recipe: {recipe}")

Recipe: config_groups=None targets=['Linear'] ignore=['lm_head'] scheme='W4A16' kv_cache_scheme=None weight_observer=None input_observer=None output_observer=None observer=None bypass_divisibility_checks=False requires_calibration_data=True index=None group=None start=None end=None update=None initialized_=False finalized_=False started_=False ended_=False block_size=128 dampening_frac=0.01 actorder=static batched_quantization='auto'


## Compressed Model Output Dir

In [18]:
# Define the local save directory name and target Drive path
SAVE_DIR = MODEL_ID + "-W4A16"
DRIVE_DIR = os.path.join(DRIVE_BASE_DIR, SAVE_DIR)
# os.makedirs(DRIVE_DIR, exist_ok=True)

## Dataset Preparation

In [36]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
# print(ds, type(ds), len(ds))
# print(ds[1]['text'])
ds = ds.filter(lambda x: len(x["text"].strip()) > 0)
# len(ds)
ds = ds.shuffle(seed=42).select(range(256))

def tokenize(sample):
    return tokenizer(
        sample["text"],
        # padding=False,
        # truncation=True,
        # max_length=4096,
        # add_special_tokens=False,
    )

# ds = ds.map(tokenize, remove_columns=ds.column_names)
ds = ds.map(tokenize)
print(ds)

Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 256
})


## Running Compression

In [38]:
from llmcompressor import oneshot

oneshot(
    model=MODEL_ID,
    tokenizer=tokenizer,
    dataset=ds,
    recipe=recipe,
    output_dir=DRIVE_DIR,
    max_seq_length=4096,
    num_calibration_samples=256,

)
print(f"Quantization complete. Model saved to: {DRIVE_DIR}")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

2026-11-20T12:12:47.3423 | reset | INFO - Compression lifecycle reset
2026-11-20T12:12:47.3563 | from_modifiers | INFO - Creating recipe from modifiers


Applying quantization config: 100%|██████████| 196/196 [00:00<00:00, 3613.73it/s]

2026-11-20T12:12:47.4325 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-11-20T12:12:47.4334 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`



/usr/local/lib/python3.13/dist-packages/llmcompressor/pipelines/sequential/helpers.py:686: FutureWarning: Found persistent modules outside of subgraphs. This feature will be deprecated in a future release.
  return func(*args, **kwargs)
(2/29): Calibrating: 100%|██████████| 256/256 [00:02<00:00, 111.67it/s]


2026-11-20T12:12:51.5691 | compress_module_list | INFO - Quantizing 1 module(s): ['model.layers.0.self_attn.q_proj']
2026-11-20T12:12:57.3937 | GPTQ | METRIC - time 5.82s
2026-11-20T12:12:57.3948 | GPTQ | METRIC - error 35.08
2026-11-20T12:12:57.3964 | GPTQ | METRIC - Accelerator 0 | usage: 7.98% | total memory: 15.6 Gb
2026-11-20T12:12:57.4008 | compress_module_list | INFO - Quantizing 2 module(s): ['model.layers.0.self_attn.k_proj', 'model.layers.0.self_attn.v_proj']
2026-11-20T12:12:57.4307 | GPTQ | METRIC - time 0.03s
2026-11-20T12:12:57.4314 | GPTQ | METRIC - error 11.83
2026-11-20T12:12:57.4326 | GPTQ | METRIC - Accelerator 0 | usage: 7.98% | total memory: 15.6 Gb
2026-11-20T12:12:57.4331 | GPTQ | METRIC - time 0.03s
2026-11-20T12:12:57.4337 | GPTQ | METRIC - error 15.46
2026-11-20T12:12:57.4352 | GPTQ | METRIC - Accelerator 0 | usage: 7.98% | total memory: 15.6 Gb
2026-11-20T12:12:57.4373 | compress_module_list | INFO - Quantizing 1 module(s): ['model.layers.0.self_attn.o_proj']

... (77 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-11-20T12:14:21.3083 | resave_config | INFO - Resaved original config with patched fields to /content/drive/MyDrive/llm_compressor_practice/Qwen/Qwen3-0.6B-W4A16/config.json


Dispatching model: 100%|██████████| 427/427 [00:00<00:00, 15202.69it/s]

Quantization complete. Model saved to: /content/drive/MyDrive/llm_compressor_practice/Qwen/Qwen3-0.6B-W4A16


## Model size Comparison
- Before and after compression

In [39]:
import pathlib

def folder_size(path):
    p = pathlib.Path(path)
    if not p.exists():
        return 0
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file())

def format_size(nbytes):
    if nbytes < 1024**2:
        return f"{nbytes/1024:.1f} KB"
    if nbytes < 1024**3:
        return f"{nbytes/1024**2:.1f} MB"
    return f"{nbytes/1024**3:.2f} GB"

size_orig = folder_size(BASE_MODEL_SAVE_DIR)
size_q = folder_size(DRIVE_DIR)
reduction = (1 - size_q / size_orig) * 100 if size_orig > 0 else 0

print("Model Size Comparison")
print("=" * 45)
print(f"Original (BF16):    {format_size(size_orig)}")
print(f"Quantized (W4A16):  {format_size(size_q)}")
print(f"Reduction:          {reduction:.0f}%")

Model Size Comparison
Original (BF16):    1.12 GB
Quantized (W4A16):  524.4 MB
Reduction:          54%


## Response Generation

### Base Model

In [42]:
import torch

prompt = "Machine learning is a branch of"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_SAVE_DIR)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_SAVE_DIR,
    device_map="cuda",
    dtype=torch.bfloat16,
)

inputs = tokenizer(prompt, return_tensors="pt")
outputs = base_model.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)
print(outputs)
generated = outputs[0][inputs["input_ids"].shape[-1]:]
print(generated)

print(f"Base Model ({BASE_MODEL_SAVE_DIR})")
print(f"Prompt: {prompt}")
print(f"Response: {tokenizer.decode(generated, skip_special_tokens=True)}")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

tensor([[21605,  6832,   374,   264,  8870,   315, 20443, 11229,   429,   702,
         18163,  5089,  6529,   304,  3213,  1635,    11,  7945,   304,   279,
          2266,   315,   279, 10000,   315,  2409,   821,   323,   279,  1184,
           369, 11050,    11, 68211,  9904,   311,  6351,  5322,    13,  1634,
           279,  2070,  9539,   311, 37580,    11,   279, 17590,   315,  5662,
          6832,  1119,  5257, 19102,   374, 10454, 14756, 23782,    13,  4354,
            11,  8818,  1181,  7826, 22538,    11]])
tensor([20443, 11229,   429,   702, 18163,  5089,  6529,   304,  3213,  1635,
           11,  7945,   304,   279,  2266,   315,   279, 10000,   315,  2409,
          821,   323,   279,  1184,   369, 11050,    11, 68211,  9904,   311,
         6351,  5322,    13,  1634,   279,  2070,  9539,   311, 37580,    11,
          279, 17590,   315,  5662,  6832,  1119,  5257, 19102,   374, 10454,
        14756, 23782,    13,  4354,    11,  8818,  1181,  7826, 22538,    11])
Base

### Quantized Model

In [43]:
quant_model = AutoModelForCausalLM.from_pretrained(
    DRIVE_DIR,
    device_map="cuda",
    dtype=torch.bfloat16,
)

inputs = tokenizer(prompt, return_tensors="pt")
outputs = quant_model.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)
generated = outputs[0][inputs["input_ids"].shape[-1]:]

print(f"Quantized Model ({DRIVE_DIR})")
print(f"Prompt: {prompt}")
print(f"Response: {tokenizer.decode(generated, skip_special_tokens=True)}")

Compressing model: 100%|██████████| 196/196 [00:00<00:00, 10016.61it/s]


Loading weights:   0%|          | 0/702 [00:00<?, ?it/s]

Decompressing model: 100%|██████████| 196/196 [00:00<00:00, 1436.04it/s]


Quantized Model (/content/drive/MyDrive/llm_compressor_practice/Qwen/Qwen3-0.6B-W4A16)
Prompt: Machine learning is a branch of
Response:  artificial intelligence that has become increasingly popular in the field of machine learning. It is a powerful tool that can be used to solve a wide range of problems. The main idea of machine learning is to use a model that can be trained to make predictions based on data. The model is trained using a set


## Perplexity Calculation

### Calculation Method

In [54]:
from datasets import load_dataset
import math
import torch

def calculate_perplexity(model, tokenizer, dataset, max_tokens=5000, stride=512):
    # Get the model's device
    device = next(model.parameters()).device

    encodings = tokenizer(
        "\n\n".join(dataset["text"]),
        return_tensors="pt", truncation=True, max_length=max_tokens,
    )
    input_ids = encodings.input_ids
    nlls, prev_end = [], 0

    for begin_loc in range(0, input_ids.size(1), stride):
        end_loc = min(begin_loc + stride, input_ids.size(1))
        trg_len = end_loc - prev_end
        input_slice = input_ids[:, begin_loc:end_loc].to(device)
        target_slice = input_slice.clone()
        target_slice[:, :-trg_len] = -100
        with torch.no_grad():
            loss = model(input_slice, labels=target_slice).loss
            nlls.append(loss * trg_len)
        prev_end = end_loc

    return math.exp(torch.stack(nlls).sum() / prev_end)

test_data = load_dataset("salesforce/wikitext", "wikitext-2-raw-v1", split="test")
print(f"Loaded {len(test_data)} test samples")

Loaded 4358 test samples


### Quantized Model

In [52]:
quant_ppl = calculate_perplexity(quant_model, tokenizer, test_data)
print(f"Quantized perplexity: {quant_ppl:.2f}")

Quantized perplexity: 34.88


## Base Model

In [56]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_SAVE_DIR, device_map="cuda", dtype=torch.bfloat16,
)
base_ppl = calculate_perplexity(base_model, tokenizer, test_data)
print(f"Base perplexity: {base_ppl:.2f}")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Base perplexity: 32.77


### Calculation

In [57]:
print("Perplexity Comparison")
print("=" * 40)
print(f"Base (BF16):      {base_ppl:.2f}")
print(f"Quantized (W4A16): {quant_ppl:.2f}")
print(f"Difference:       {quant_ppl - base_ppl:+.2f} ({(quant_ppl/base_ppl - 1)*100:+.1f}%)")
print(f"\nA small increase in perplexity is expected — the quantized layers use 4-bit weights.")

Perplexity Comparison
Base (BF16):      32.77
Quantized (W4A16): 34.88
Difference:       +2.11 (+6.4%)

A small increase in perplexity is expected — the quantized layers use 4-bit weights.
